In [11]:
from pathlib import Path
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: chunking_test_document.pdf
  ✓ Loaded 3 pages

Processing: TCS.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-16T16:49:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-16T16:49:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\chunking_test_document.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'chunking_test_document.pdf', 'file_type': 'pdf'}, page_content="Page 1\n PDF TEXT EXTRACTION & CHUNKING TEST\nA sample document for testing PDF loaders, text extraction, and text chunking\nDocument Information\nDocument ID: CHUNK-TEST-001\nVersion: 1.0\nPrepared: November 2026\nPurpose: This document contains headings, paragraphs, lists, a table, and multiple sections so that a\nPDF-processing pipeline can be tested with different content structures.\n1. Introduction to Document Chunking\nDocument chunking is the process of splitting a long document into smaller pieces of tex

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_documents(documents, chunk_size=500, chunk_overlap=50):

    if not documents:
        print("No documents received!")
        return []

    # Convert raw strings into Document objects
    if isinstance(documents[0], str):
        documents = [
            Document(page_content=doc, metadata={})
            for doc in documents
        ]

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    print("Number of documents:", len(all_pdf_documents))

    print("Document content:")
    print(repr(all_pdf_documents[0].page_content))

    print("Content length:")
    print(len(all_pdf_documents[0].page_content))

    split_docs = text_splitter.split_documents(documents)

    print(f"Input documents: {len(documents)}")
    print(f"Total chunks created: {len(split_docs)}")

    for i, doc in enumerate(split_docs[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Length: {len(doc.page_content)}")
        print(f"Content:\n{doc.page_content}")
        print(f"Metadata: {doc.metadata}")

    return split_docs


chunks = split_documents(all_pdf_documents)

Number of documents: 4
Document content:
"Page 1\n PDF TEXT EXTRACTION & CHUNKING TEST\nA sample document for testing PDF loaders, text extraction, and text chunking\nDocument Information\nDocument ID: CHUNK-TEST-001\nVersion: 1.0\nPrepared: November 2026\nPurpose: This document contains headings, paragraphs, lists, a table, and multiple sections so that a\nPDF-processing pipeline can be tested with different content structures.\n1. Introduction to Document Chunking\nDocument chunking is the process of splitting a long document into smaller pieces of text. These pieces are\ncommonly called chunks. Chunking is useful in retrieval-augmented generation (RAG) systems because a\nlanguage model can retrieve only the sections that are relevant to a user's question instead of processing an\nentire document every time.\nA good chunking strategy should preserve meaning. For example, a paragraph explaining a single concept\nshould ideally remain together. If a chunk ends in the middle of a senten

### Embedding and VectorStore DB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3203.33it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\VEDASHREE V\AppData\Local\Temp\ipykernel_10932\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore DB

In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [18]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-16T16:49:23+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-16T16:49:23+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\chunking_test_document.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'chunking_test_document.pdf', 'file_type': 'pdf'}, page_content='Page 1\n PDF TEXT EXTRACTION & CHUNKING TEST\nA sample document for testing PDF loaders, text extraction, and text chunking\nDocument Information\nDocument ID: CHUNK-TEST-001\nVersion: 1.0\nPrepared: November 2026\nPurpose: This document contains headings, paragraphs, lists, a table, and multiple sections so that a\nPDF-processing pipeline can be tested with different content structures.\n1. Introduction to Document Chunking'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(u

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]


Generated embeddings with shape: (13, 384)
Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 26


In [20]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]